<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# DIY: Data Preprocessing and Feature Engineering

Practice encoding strategies on **two new datasets** and discover common pitfalls that arise with high-cardinality categorical features.

This notebook builds on what you learned in *Day 1 — Data Preprocessing and Feature Engineering*.

---

### Table of Contents

- [1. Load Datasets](#load)
- [2. OHE Dimensionality Explosion](#ohe-explosion)
- [3. OHE and Overfitting](#ohe-overfitting)
- [4. Frequency Encoding Collision](#freq-collision)

---

## Setup

In [ ]:
import os
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('tableau-colorblind10')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder

# ── helpers ──
def show(df, n=5):
    display(df.head(n))

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

print("Environment ready.")

---

## <a id="load"></a> Section 1 — Load Datasets

We use two datasets from OpenML (no download or authentication required):

| Dataset | Rows | Features | Task | Key property |
|---|---:|---:|---|---|
| **Amazon Employee Access** | ~32 K | 9 categorical | Binary classification (access granted/denied) | Extreme cardinality (up to 5 946 unique values per feature) |
| **King County House Sales** | ~21 K | 19 mixed | Regression (house price) | Geographic `zipcode` feature with 70 levels |

In [ ]:
# ── Amazon Employee Access (OpenML ID 4135) ──
amazon_raw = fetch_openml(data_id=4135, as_frame=True, parser="auto")
amazon = amazon_raw.frame

print("Amazon Employee Access")
print(f"  Shape: {amazon.shape}")
print(f"  Target: '{amazon_raw.target.name}' — positive rate: {amazon_raw.target.astype(int).mean():.4f}")
print(f"  Feature dtypes: {amazon.dtypes.value_counts().to_dict()}")
print()
show(amazon)

In [ ]:
# ── King County House Sales (OpenML ID 42092) ──
kc_raw = fetch_openml(data_id=42092, as_frame=True, parser="auto")
kc = kc_raw.frame

print("King County House Sales")
print(f"  Shape: {kc.shape}")
print(f"  Target: '{kc_raw.target.name}' — median: ${kc_raw.target.astype(float).median():,.0f}")
print()
show(kc)

In [ ]:
# ── Cardinality overview for Amazon ──
print("Unique values per feature (Amazon):")
for col in amazon.columns:
    if col != amazon_raw.target.name:
        print(f"  {col:30s} {amazon[col].nunique():>6,}")

---

## <a id="ohe-explosion"></a> Section 2 — OHE Dimensionality Explosion (Amazon)

One-hot encoding (OHE) creates one binary column per unique category value.
When a feature has thousands of levels, this can produce a matrix with **more columns than rows**.

**Why this matters:**
- Linear models (logistic regression, linear SVM) struggle when *p >> n* — the system is underdetermined and the coefficient matrix cannot be reliably inverted.
- Memory consumption grows linearly with the number of columns.
- Even tree-based models slow down significantly with extremely wide sparse inputs.

In [ ]:
# ── OHE on a single high-cardinality feature ──
feature_col = "RESOURCE"
n_unique = amazon[feature_col].nunique()
print(f"'{feature_col}' has {n_unique:,} unique values")
print(f"Dataset has {len(amazon):,} rows")
print(f"Ratio (columns / rows) after OHE on this single feature: {n_unique / len(amazon):.2f}")
print()

In [ ]:
# ── OHE on ALL categorical features ──
feature_cols = [c for c in amazon.columns if c != amazon_raw.target.name]

ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_ohe = ohe.fit_transform(amazon[feature_cols])

print(f"Original shape:  {amazon[feature_cols].shape}")
print(f"OHE shape:       {X_ohe.shape}")
print(f"Columns / rows:  {X_ohe.shape[1] / X_ohe.shape[0]:.2f}")
print(f"Sparsity:        {1 - X_ohe.nnz / (X_ohe.shape[0] * X_ohe.shape[1]):.6f}")

> **Do It Yourself**
>
> Look at the cardinality of each feature in the Amazon dataset.
> Which features would you one-hot encode? Which would you not? Why?
> What alternative encoding would you use for the high-cardinality ones?

In [ ]:
# Your exploration here


---

## <a id="ohe-overfitting"></a> Section 3 — OHE and Overfitting (Amazon)

Even when OHE fits in memory, the resulting wide sparse matrix lets linear models **memorize training noise**.
With thousands of binary columns and relatively few rows, the model has enough degrees of freedom to fit the training set almost perfectly — but this does not generalize.

We will compare:
1. **OHE + Logistic Regression** — expect a large train/test AUC gap.
2. **Frequency Encoding + Logistic Regression** — expect a smaller gap.

In [ ]:
# ── Prepare data ──
feature_cols = [c for c in amazon.columns if c != amazon_raw.target.name]
y = amazon_raw.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    amazon[feature_cols], y, test_size=0.3, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")

In [ ]:
# ── Approach 1: OHE ──
ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_train_ohe = ohe.fit_transform(X_train)
X_test_ohe = ohe.transform(X_test)

lr_ohe = LogisticRegression(max_iter=300, random_state=SEED, solver="saga")
lr_ohe.fit(X_train_ohe, y_train)

train_auc_ohe = roc_auc_score(y_train, lr_ohe.predict_proba(X_train_ohe)[:, 1])
test_auc_ohe = roc_auc_score(y_test, lr_ohe.predict_proba(X_test_ohe)[:, 1])

print("OHE + Logistic Regression")
print(f"  Train AUC: {train_auc_ohe:.4f}")
print(f"  Test AUC:  {test_auc_ohe:.4f}")
print(f"  Gap:       {train_auc_ohe - test_auc_ohe:.4f}")

In [ ]:
# ── Approach 2: Frequency Encoding ──
X_train_freq = X_train.copy()
X_test_freq = X_test.copy()

for col in feature_cols:
    freq_map = X_train[col].value_counts(normalize=True)
    X_train_freq[col] = X_train[col].map(freq_map).astype(float)
    X_test_freq[col] = X_test[col].map(freq_map).fillna(0).astype(float)

lr_freq = LogisticRegression(max_iter=300, random_state=SEED, solver="saga")
lr_freq.fit(X_train_freq, y_train)

train_auc_freq = roc_auc_score(y_train, lr_freq.predict_proba(X_train_freq)[:, 1])
test_auc_freq = roc_auc_score(y_test, lr_freq.predict_proba(X_test_freq)[:, 1])

print("Frequency Encoding + Logistic Regression")
print(f"  Train AUC: {train_auc_freq:.4f}")
print(f"  Test AUC:  {test_auc_freq:.4f}")
print(f"  Gap:       {train_auc_freq - test_auc_freq:.4f}")

In [ ]:
# ── Side-by-side comparison ──
comparison = pd.DataFrame({
    "Encoding": ["OHE", "Frequency"],
    "Train AUC": [train_auc_ohe, train_auc_freq],
    "Test AUC": [test_auc_ohe, test_auc_freq],
    "Gap": [train_auc_ohe - test_auc_ohe, train_auc_freq - test_auc_freq],
})
display(comparison)

> **Do It Yourself**
>
> Try **target encoding** on the `RESOURCE` feature:
> - For each category, compute the mean of the target in the training set.
> - Map these means onto both train and test.
> - Train a `LogisticRegression` and compute train/test AUC.
>
> Does the train/test gap shrink compared to OHE? Compared to frequency encoding?
>
> *Hint: be careful about leakage — compute the means on the training set only.*

In [ ]:
# Your target encoding experiment here


---

## <a id="freq-collision"></a> Section 4 — Frequency Encoding Collision (King County)

Frequency encoding replaces each category with its occurrence count (or proportion).
This is compact and avoids the OHE explosion — but it has a subtle flaw:

**Categories with similar frequencies get mapped to the same (or nearly the same) value, even if their relationship to the target is completely different.**

We will demonstrate this with the `zipcode` feature in the King County house sales dataset, where different neighborhoods can have similar numbers of sales but very different house prices.

In [ ]:
# ── Zipcode summary: frequency vs median price ──
kc["price"] = kc[kc_raw.target.name].astype(float)

zip_stats = (
    kc.groupby("zipcode")
    .agg(count=("price", "size"), median_price=("price", "median"))
    .sort_values("count", ascending=False)
    .reset_index()
)

print(f"Number of zipcodes: {zip_stats.shape[0]}")
print(f"Sales per zipcode — min: {zip_stats['count'].min()}, max: {zip_stats['count'].max()}, median: {zip_stats['count'].median():.0f}")
print()
show(zip_stats, n=10)

In [ ]:
# ── Find collision pairs: similar frequency, very different prices ──
zip_stats_sorted = zip_stats.sort_values("count").reset_index(drop=True)

collisions = []
for i in range(len(zip_stats_sorted)):
    for j in range(i + 1, len(zip_stats_sorted)):
        row_i = zip_stats_sorted.iloc[i]
        row_j = zip_stats_sorted.iloc[j]
        freq_diff = abs(row_i["count"] - row_j["count"])
        price_ratio = max(row_i["median_price"], row_j["median_price"]) / min(row_i["median_price"], row_j["median_price"])
        if freq_diff <= 5 and price_ratio >= 1.8:
            collisions.append({
                "zipcode_A": row_i["zipcode"],
                "count_A": row_i["count"],
                "median_price_A": row_i["median_price"],
                "zipcode_B": row_j["zipcode"],
                "count_B": row_j["count"],
                "median_price_B": row_j["median_price"],
                "price_ratio": price_ratio,
            })

collisions_df = pd.DataFrame(collisions).sort_values("price_ratio", ascending=False)
print(f"Found {len(collisions_df)} zipcode pairs with similar frequency but different prices (ratio >= 1.8x):")
print()
show(collisions_df, n=10)

In [ ]:
# ── Visualize: frequency vs median price scatter ──
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(zip_stats["count"], zip_stats["median_price"] / 1000, alpha=0.7, edgecolors="k", linewidths=0.5)
ax.set_xlabel("Number of sales (= frequency encoding value)")
ax.set_ylabel("Median price ($K)")
ax.set_title("Frequency Encoding Collision: same x-axis value, wildly different y-axis")

# Annotate a collision pair if available
if len(collisions_df) > 0:
    top = collisions_df.iloc[0]
    for label, count_col, price_col in [("A", "count_A", "median_price_A"), ("B", "count_B", "median_price_B")]:
        ax.annotate(
            f"zip {top[f'zipcode_{label}']}",
            xy=(top[count_col], top[price_col] / 1000),
            fontsize=9, fontweight="bold", color="red",
            arrowprops=dict(arrowstyle="->", color="red"),
            xytext=(top[count_col] + 20, top[price_col] / 1000 + 50),
        )

plt.tight_layout()
plt.show()

In [ ]:
# ── After frequency encoding, these zipcodes become indistinguishable ──
if len(collisions_df) > 0:
    top = collisions_df.iloc[0]
    freq_map = kc["zipcode"].value_counts(normalize=True)
    print(f"Zipcode {top['zipcode_A']}:")
    print(f"  Frequency-encoded value: {freq_map[top['zipcode_A']]:.6f}")
    print(f"  Median price: ${top['median_price_A']:,.0f}")
    print()
    print(f"Zipcode {top['zipcode_B']}:")
    print(f"  Frequency-encoded value: {freq_map[top['zipcode_B']]:.6f}")
    print(f"  Median price: ${top['median_price_B']:,.0f}")
    print()
    print(f"Frequency difference: {abs(freq_map[top['zipcode_A']] - freq_map[top['zipcode_B']]):.6f}")
    print(f"Price ratio: {top['price_ratio']:.1f}x")
    print()
    print("A model using frequency encoding treats these two neighborhoods as nearly identical.")

> **Do It Yourself**
>
> What encoding strategy would preserve the price signal for `zipcode`?
>
> Try one or more of these approaches and compare:
> - **Target encoding**: replace each zipcode with the mean price in the training set.
> - **Ordinal encoding by target**: rank zipcodes by mean price and use the rank.
> - **Binning**: group zipcodes into price tiers (e.g., low / medium / high) and one-hot encode the tiers.
>
> *Hint: remember to compute encodings on the training set only to avoid leakage.*

In [ ]:
# Your encoding experiment here


---

## Scratch Space

In [ ]:
# Free exploration
